# Study Buddy — Gemma 2B Fine-tuning on Kaggle

**What this notebook does:**
1. Installs / upgrades required packages
2. Authenticates with Hugging Face (Gemma is a gated model — you need a HF token)
3. Loads our 99-example Study Buddy JSONL dataset
4. Loads `google/gemma-2b-it` in 4-bit precision (QLoRA)
5. Applies LoRA adapters via PEFT
6. Fine-tunes with TRL's SFTTrainer
7. Runs a quick inference test
8. Saves the adapter weights

**Before running:**
- Enable GPU: *Settings → Accelerator → GPU T4 x2*
- Add your HF token: *Add-ons → Secrets → New Secret* → name `HF_TOKEN`, value = your token from https://huggingface.co/settings/tokens
- Upload `study_buddy_finetune.jsonl` as a Kaggle Dataset and attach it to this notebook

In [ ]:
# ── Cell 1: Install / upgrade packages ───────────────────────────────────────
# Kaggle has older versions pre-installed; we need recent TRL and PEFT.
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "transformers>=4.40",
    "datasets>=2.18",
    "trl>=0.8",
    "peft>=0.10",
    "accelerate>=0.28",
    "bitsandbytes>=0.43",
], check=True)

print("Packages ready.")

In [ ]:
# ── Cell 2: Authenticate with Hugging Face ────────────────────────────────────
# Gemma is a gated model. You must:
#   1. Accept the licence at https://huggingface.co/google/gemma-2b-it
#   2. Store your HF token in Kaggle Secrets as HF_TOKEN
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
print("Logged in to Hugging Face.")

In [ ]:
# ── Cell 3: Verify GPU ────────────────────────────────────────────────────────
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {name}  ({mem:.1f} GB)")

In [ ]:
# ── Cell 4: Load dataset ──────────────────────────────────────────────────────
# Update DATASET_PATH to wherever your JSONL file lives in the Kaggle input.
# Typical path after attaching a Kaggle Dataset:
#   /kaggle/input/<your-dataset-name>/study_buddy_finetune.jsonl
import json
from pathlib import Path

DATASET_PATH = "/kaggle/input/study-buddy-dataset/study_buddy_finetune.jsonl"

records = []
with open(DATASET_PATH) as f:
    for line in f:
        records.append(json.loads(line))

print(f"Loaded {len(records)} examples")
print("Sample instruction:", records[0]["instruction"])

In [ ]:
# ── Cell 5: Format prompts using Gemma-IT chat template ───────────────────────
# Gemma instruction-tuned models expect this exact format:
#   <start_of_turn>user\n{instruction}<end_of_turn>\n<start_of_turn>model\n{output}<end_of_turn>

def format_prompt(example: dict) -> str:
    instruction = example["instruction"]
    if example.get("input"):
        instruction = f"{instruction}\n\n{example['input']}"
    return (
        f"<start_of_turn>user\n{instruction}<end_of_turn>\n"
        f"<start_of_turn>model\n{example['output']}<end_of_turn>"
    )

# Preview one formatted prompt
print(format_prompt(records[0]))

In [ ]:
# ── Cell 6: Build HuggingFace Dataset ────────────────────────────────────────
from datasets import Dataset

formatted = [{"text": format_prompt(r)} for r in records]
hf_dataset = Dataset.from_list(formatted)

# 90 / 10 train-eval split
split = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"Train: {len(train_dataset)}  Eval: {len(eval_dataset)}")

In [ ]:
# ── Cell 7: Load tokenizer ────────────────────────────────────────────────────
from transformers import AutoTokenizer

MODEL_ID = "google/gemma-2b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token   # Gemma has no pad token by default
tokenizer.padding_side = "right"            # Required for SFTTrainer

print("Tokenizer loaded. Vocab size:", tokenizer.vocab_size)

In [ ]:
# ── Cell 8: Load model in 4-bit (QLoRA) ──────────────────────────────────────
# BitsAndBytesConfig reduces Gemma 2B from ~10 GB to ~2.5 GB GPU memory.
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",            # NormalFloat4 — best quality 4-bit
    bnb_4bit_compute_dtype=torch.bfloat16, # Compute in bf16 for stability
    bnb_4bit_use_double_quant=True,        # Nested quantisation — saves ~0.4 GB
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",                     # Spread across available GPUs
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False             # Disable KV cache during training

print("Model loaded.")
print(f"Memory allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── Cell 9: Apply LoRA adapters ───────────────────────────────────────────────
# LoRA injects small trainable matrices into the attention layers.
# We only train ~0.5 % of parameters instead of all 2 billion.
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)  # Enable gradient checkpointing

lora_config = LoraConfig(
    r=16,                  # Rank — higher = more capacity, more memory
    lora_alpha=32,         # Scaling factor (alpha / r = effective learning rate)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── Cell 10: Configure training ───────────────────────────────────────────────
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/kaggle/working/gemma-study-buddy",
    num_train_epochs=5,               # Small dataset — more epochs compensate
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,    # Effective batch size = 4 × 4 = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,                        # Use bfloat16 on T4
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",                 # Disable wandb
    optim="paged_adamw_8bit",         # Memory-efficient optimiser
)

In [ ]:
# ── Cell 11: Run SFTTrainer ───────────────────────────────────────────────────
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=512,
    packing=False,
)

print("Starting fine-tuning ...")
trainer.train()
print("Training complete.")

In [ ]:
# ── Cell 12: Quick inference test ─────────────────────────────────────────────
# Check the fine-tuned model responds sensibly before saving.
model.eval()

def ask(instruction: str, max_new_tokens: int = 300) -> str:
    prompt = f"<start_of_turn>user\n{instruction}<end_of_turn>\n<start_of_turn>model\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Decode only the newly generated tokens
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


print("=" * 60)
print("TEST 1 — Concept explanation")
print(ask("Explain what a p-value is in simple terms."))

print("=" * 60)
print("TEST 2 — Flashcard generation")
print(ask("Create 5 flashcards on Python lists vs tuples."))

print("=" * 60)
print("TEST 3 — Study advice")
print(ask("I have 2 weeks before my exam. How should I study?"))

In [ ]:
# ── Cell 13: Save adapter weights ─────────────────────────────────────────────
# We save only the LoRA adapter — much smaller than the full model.
# To use later: load base Gemma + merge adapter via PeftModel.from_pretrained.
SAVE_PATH = "/kaggle/working/gemma-study-buddy-adapter"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

import os
size_mb = sum(
    os.path.getsize(os.path.join(SAVE_PATH, f))
    for f in os.listdir(SAVE_PATH)
) / 1e6
print(f"Adapter saved to {SAVE_PATH}  ({size_mb:.1f} MB)")

In [ ]:
# ── Cell 14 (optional): Merge adapter into base model and save full model ─────
# Only run this if you want a standalone merged model (larger file).
# The adapter-only save from Cell 13 is enough for the Gradio app.

from peft import PeftModel
from transformers import AutoModelForCausalLM
import torch

MERGED_PATH = "/kaggle/working/gemma-study-buddy-merged"

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
)
merged = PeftModel.from_pretrained(base, SAVE_PATH)
merged = merged.merge_and_unload()  # Fold LoRA weights into base
merged.save_pretrained(MERGED_PATH)
tokenizer.save_pretrained(MERGED_PATH)
print(f"Merged model saved to {MERGED_PATH}")